# 10 — Subject-disjoint C-NMC splits

**Purpose:** Freeze fold_2 as test and stratify subjects in folds 0+1 for train/validation.

**Required inputs:** Validated cnmc_manifest.csv from notebook 09.

**Saved outputs:** outputs/classifier/splits/{train,val,test,excluded}.csv, summary.json and leakage_report.json.

## 1. Load central configuration

Set `cnmc_root` once in ignored `config.local.yaml`. Source images are never moved or changed.

In [7]:
from pathlib import Path
import os, sys
search = [Path.cwd(), *Path.cwd().parents]
if os.environ.get("LEUKEMIA_YOLO_ROOT"):
    search.insert(0, Path(os.environ["LEUKEMIA_YOLO_ROOT"]))
for candidate in search:
    if (candidate / "config.example.yaml").is_file() and (candidate / "src" / "classifier_config.py").is_file():
        sys.path.insert(0, str(candidate)); break
from src.classifier_config import classifier_config, classifier_output
cfg = classifier_config()
out = classifier_output(cfg)
print("Project:", cfg["root"])
print("Config:", cfg["config_path"])
print("C-NMC source:", cfg["cnmc_root"])

Project: D:\CellCancer\leukemia-yolo
Config: D:\CellCancer\leukemia-yolo\config.local.yaml
C-NMC source: D:\Datasets\C-NMC_Leukemia


## 2. Load source manifest

Only labeled training folds are included; preliminary/final challenge images are excluded.

In [8]:
import pandas as pd
from src.cnmc_data import inspect_cnmc
source, inspection = inspect_cnmc(cfg)
assert inspection["ready"]
display(source.groupby(["fold", "class_name"]).size().rename("cells").reset_index())

,fold,class_name,cells
0,fold_0,ALL,2397
1,fold_0,HEM,1130
2,fold_1,ALL,2418
3,fold_1,HEM,1163
4,fold_2,ALL,2457
5,fold_2,HEM,1096


## 3. Create or reuse immutable split

This cell reloads the splitter and displays its file and version so Jupyter cannot keep using the original cached function. fold_2 stays fixed; affected candidate subjects are excluded and audited. Conflicting labels stop splitting. Saved splits are reused.

In [9]:
import importlib, inspect
import src.cnmc_splits as split_module
importlib.invalidate_caches()
split_module = importlib.reload(split_module)
print("Splitter file:", Path(split_module.__file__).resolve())
print("Splitter implementation version:", getattr(split_module, "SPLIT_IMPLEMENTATION_VERSION", "old/missing"))
if (getattr(split_module, "SPLIT_IMPLEMENTATION_VERSION", None) != 2 or
        "excluded" not in inspect.signature(split_module.validate_splits).parameters):
    raise RuntimeError(f"The source file {split_module.__file__} is outdated. Copy the updated src folder "
                       "and notebook 10 into this project, restart the kernel, and rerun from the first cell.")
create_or_load_splits = split_module.create_or_load_splits
validate_splits = split_module.validate_splits
splits, summary = create_or_load_splits(cfg, source)
display(pd.DataFrame(summary["counts"]).T)
print(summary["protocol"])
print("Excluded candidate cells:", summary.get("excluded_images", 0))
print("Excluded candidate subjects:", summary.get("excluded_subjects", []))
print("Leakage audit:", out / "splits" / "leakage_report.json")
assert not (set(splits["train"].subject_id) & set(splits["val"].subject_id))

Splitter file: D:\CellCancer\leukemia-yolo\src\cnmc_splits.py
Splitter implementation version: 2
WARNING — excluded 1256 candidate cells from 4 subjects to prevent test leakage. fold_2 unchanged. Audit: D:\CellCancer\leukemia-yolo\outputs\classifier\splits\leakage_report.json


,images,subjects,normal,blast
train,4667,30,1600,3067
val,1185,8,270,915
test,3553,31,1096,2457


fold_2 fixed test; fold_0+fold_1 subject-stratified train/val; test-overlapping candidate subjects excluded; duplicate-linked subjects grouped
Excluded candidate cells: 1256
Excluded candidate subjects: ['ALL:20', 'ALL:51', 'HEM:H10', 'HEM:H6']
Leakage audit: D:\CellCancer\leukemia-yolo\outputs\classifier\splits\leakage_report.json


## 4. Leakage gate

Subjects, absolute paths and SHA-256 image content must not cross splits. Full source coverage including excluded.csv was validated during creation/loading.

In [10]:
verified = validate_splits(splits)
assert set(splits["test"].fold) == {"fold_2"}
print("PASS — subject/path/content disjoint; saved manifests point to original images")

PASS — subject/path/content disjoint; saved manifests point to original images


## Next

Open **11_EfficientNet_B3_Training.ipynb** for the CUDA smoke test and two-phase training.